In [1]:

import operator
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List
from uuid import uuid4

from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode 
from langchain_groq import ChatGroq

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma 
from langchain.tools.retriever import create_retriever_tool 
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults 


print("Carregando ambiente e importações...")
load_dotenv()

LLM = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7) 

print("Recriando as ferramentas para o agente...")

print(" -> Configurando RAG...")

CHROMA_PATH = "dados/chroma_db"
EMBEDDING_MODEL = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") 

def get_rag_retriever(chroma_path, embedding_model):
    try:
        db = Chroma(persist_directory=chroma_path, embedding_function=embedding_model)
        return db.as_retriever(search_kwargs={"k": 3})
    except Exception as e:
        print(f" AVISO: Erro ao carregar ChromaDB. Execute 01_RAG_Preparacao.ipynb. Erro: {e}")
        return None

rag_retriever = get_rag_retriever(CHROMA_PATH, EMBEDDING_MODEL)
rag_tool = create_retriever_tool(rag_retriever, "fitness_knowledge_base_search", "Busca na base de conhecimento interna por informações sobre exercícios, séries e técnicas.")



print(" -> Configurando Web Search...")
web_search_tool = TavilySearchResults(k=3, name="web_search_tool") 


print(" -> Configurando Calculadora...")
@tool
def calculator_tool(age: int, weight_kg: float, height_cm: float, goal: str) -> str:
    """Calcula o IMC e estima o gasto calórico diário para um objetivo. Requer altura em cm.""" 
    height_m = height_cm / 100
    imc = round(weight_kg / (height_m**2), 2)
    tmb = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) + 5
    if "emagrecimento" in goal.lower(): calories = tmb * 1.375 - 400
    elif "hipertrofia" in goal.lower(): calories = tmb * 1.55 + 400
    else: calories = tmb * 1.55
    return f"Cálculos de referência:\\n- IMC: {imc}\\n- Gasto Calórico Diário Estimado para '{goal}': {int(calories)} kcal."


all_tools = [calculator_tool, rag_tool, web_search_tool] 


LLM_WITH_TOOLS = LLM.bind_tools(all_tools)
print(f" Agente configurado com {len(all_tools)} ferramentas (Calc, Web, RAG).")

c:\Users\Amanda\anaconda3\envs\ai_fitness_coach\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Carregando ambiente e importações...
Recriando as ferramentas para o agente...
 -> Configurando RAG...


C:\Users\Amanda\AppData\Local\Temp\ipykernel_14128\2139469843.py:28: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  EMBEDDING_MODEL = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
C:\Users\Amanda\AppData\Local\Temp\ipykernel_14128\2139469843.py:32: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(persist_directory=chroma_path, embedding_function=embe

 -> Configurando Web Search...
 -> Configurando Calculadora...
 Agente configurado com 3 ferramentas (Calc, Web, RAG).


C:\Users\Amanda\AppData\Local\Temp\ipykernel_14128\2139469843.py:44: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(k=3, name="web_search_tool")


In [2]:
# Definição do Estado, Agente e Nódulos do LangGraph

# Definição do Estado
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

# Inicializa o LLM e o Agente com Tool Calling
llm = ChatGroq(temperature=0, model_name="llama-3.1-8b-instant")
model_with_tools = llm.bind_tools(all_tools)

# NÓ: Execução de Ferramentas
tool_node = ToolNode(all_tools) 

# NÓ: Chamada do Modelo (Pensamento/Decisão)
def call_model_node(state: AgentState):
    print("--- NÓ: CHAMANDO O MODELO (Pensamento) ---")
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Roteador Condicional
def should_continue_router(state: AgentState):
    print("--- NÓ: ROTEADOR ---")
    if state["messages"][-1].tool_calls:
        print("Decisão: Chamar ferramentas.")
        return "call_tools"
    else:
        print("Decisão: Geração Final do Plano.")
        return "end"

# Montagem e Compilação do Grafo
graph_workflow = StateGraph(AgentState)
graph_workflow.add_node("agent", call_model_node)
graph_workflow.add_node("call_tools", tool_node)
graph_workflow.set_entry_point("agent")
graph_workflow.add_conditional_edges("agent", should_continue_router, {"call_tools": "call_tools", "end": END})
graph_workflow.add_edge("call_tools", "agent")
app = graph_workflow.compile()

print("\\n Grafo LangGraph montado. Agente pronto para processamento!")

\n Grafo LangGraph montado. Agente pronto para processamento!


In [ ]:
# Execução com Dados Falsos (Foco em Hipertrofia)

# ----------------------------------------------------
#  DADOS FIXOS DE TESTE 
# ----------------------------------------------------
USER_DATA_FAKE = {
    "age": 28,             
    "weight": 60.0,        
    "height_cm": 164.0,    
    "frequency": 5,       
    "gender": "female",    
    "goal": "hipertrofia",
    "name": "Amanda"       
}
# ----------------------------------------------------

def create_initial_prompt(user_data):
    """Cria o prompt inicial que FORÇA o fluxo de trabalho obrigatório e a saída personalizada."""
    
    total_workouts = user_data['frequency'] * 3
    
    return f"""
    Aja como um personal trainer de IA especialista.
    Sua tarefa é criar um plano de treino de musculação COMPLETO e DETALHADO.

    **REQUISITO CHAVE - ALTA PRIORIDADE:** O plano deve conter **EXATAMENTE {total_workouts} treinos distintos**, organizados em 3 semanas.
    
    **ESTRUTURA DE SAÍDA FINAL (USE EXATAMENTE ESTA ORDEM E FORMATO):**
    
    1.  **SEÇÃO DE TREINOS:** Gere os {total_workouts} treinos, enumerando-os de forma sequencial (Treino 1, Treino 2, ... até o Treino {total_workouts}), organizados em blocos de Semana 1, Semana 2 e Semana 3. Use séries/repetições e exercícios apropriados (NUNCA INVENTE NOMES).
    2.  **SEÇÃO DE ANÁLISE E DICAS:** Após a lista completa de treinos, crie uma seção final chamada "Análise do Coach e Estratégia".
        * Nesta seção, use os **Cálculos** (IMC/TMB) e o **Contexto RAG** para dar conselhos personalizados sobre:
            * **Meta Calórica** (Se precisa de superávit ou déficit).
            * **Progressão** (Como aumentar a carga nas Semanas 2 e 3).
            * **Foco Técnico** (1 ou 2 dicas de execução cruciais, Ex: Retração escapular no Supino).
    
    **Informações do Usuário:**
    - Objetivo: {user_data['goal']} | Frequência: {user_data['frequency']} dias/semana
    - Idade: {user_data['age']} | Peso: {user_data['weight']} | Altura: {user_data['height_cm']} | Gênero: {user_data['gender']}

    **Processo Obrigatório (Para o LangGraph):**
    1.  Execute CADA UMA das 3 ferramentas.
    2.  Após as ferramentas, PARE e gere a **ESTRUTURA DE SAÍDA FINAL** completa.
    """

# --- INÍCIO DA EXECUÇÃO ---

# 1. Prepara o prompt com os dados fixos
prompt = create_initial_prompt(USER_DATA_FAKE)
messages = [HumanMessage(content=prompt)]

print("\n--- INICIANDO AGENTE AI FITNESS COACH ---")
print(f" Perfil Fictício: {USER_DATA_FAKE['goal']} | Frequência: {USER_DATA_FAKE['frequency']} dias/semana")
print("-----------------------------------------")

# 2. Configura a execução
config = {"configurable": {"thread_id": str(uuid4())}}
final_plan_content = ""

# 3. Loop de Execução
for event in app.stream({"messages": messages}, config=config):
    for key, value in event.items():
        if key == "agent":
            agent_response = value['messages'][-1]
            if agent_response.tool_calls:
                print(f" CHAMA FERRAMENTA(S): {[tc['name'] for tc in agent_response.tool_calls]}") 
            else:
                final_plan_content = agent_response.content
                print(" GERAÇÃO FINAL DO PLANO.")
        elif key == "call_tools":
            tool_messages = value['messages']
            for tool_message in tool_messages:
                print(f"Resultado de {tool_message.name}: Sucesso. Dados inseridos no contexto.")

# 4. Saída Final
print("\n" + "="*70)
print(f"PLANO DE TREINO FINAL GERADO):")
print("="*70)

with open("plano_de_treino_final.md", "w", encoding="utf-8") as f:
    f.write(final_plan_content)
print("\n O plano também foi salvo em 'plano_de_treino_final.md'")


--- INICIANDO AGENTE AI FITNESS COACH ---
 Perfil Fictício: hipertrofia | Frequência: 5 dias/semana
-----------------------------------------
--- NÓ: CHAMANDO O MODELO (Pensamento) ---
--- NÓ: ROTEADOR ---
Decisão: Chamar ferramentas.
 CHAMA FERRAMENTA(S): ['fitness_knowledge_base_search', 'calculator_tool', 'web_search_tool']
Resultado de fitness_knowledge_base_search: Sucesso. Dados inseridos no contexto.
Resultado de calculator_tool: Sucesso. Dados inseridos no contexto.
Resultado de web_search_tool: Sucesso. Dados inseridos no contexto.
--- NÓ: CHAMANDO O MODELO (Pensamento) ---
--- NÓ: ROTEADOR ---
Decisão: Geração Final do Plano.
 GERAÇÃO FINAL DO PLANO.

PLANO DE TREINO FINAL GERADO):

 O plano também foi salvo em 'plano_de_treino_final.md'
